# MeetBuddy AI - Permanent GPU Transcriber

This version is pre-configured with your **Static Domain**.

### 🚀 Instructions:
1. Go to **Runtime -> Change runtime type**, select **T4 GPU** and save.
2. Paste your **Ngrok Authtoken** below.
3. Click **Run All** (Ctrl+F9).
4. Your MeetBuddy app will connect automatically to the URL printed at the bottom.

In [ ]:
# @title 🔑 Setup Your Connection
NGROK_TOKEN = "3A92MhD0p3aag5iFAH4Dr3xurXf_4ALEXg4Lv9XsS3AsrE9Bs" # @param {type:"string"}
NGROK_DOMAIN = "polysyllogistic-in-unrhapsodically.ngrok-free.dev"

print("Installing dependencies...")
!apt-get update -y && apt-get install -y zstd pciutils && pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok nest-asyncio requests

# Install Ollama
import subprocess, time
print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.ai/install.sh | sh", shell=True, check=True)

print("Starting Ollama serve...")
subprocess.Popen(['ollama', 'serve'])
time.sleep(4)

print("Pulling llama3.2 model (skip if already pulled)...")
subprocess.run(['ollama', 'pull', 'llama3.2'], check=True)
print("Ollama is ready!")

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print("Ngrok token set!")


In [ ]:
import nest_asyncio
import subprocess
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import uvicorn
from faster_whisper import WhisperModel
import os, threading, time

# ── Config (self-contained so this cell always works) ─────────────────────────
NGROK_TOKEN = "3A92MhD0p3aag5iFAH4Dr3xurXf_4ALEXg4Lv9XsS3AsrE9Bs"
NGROK_DOMAIN = "polysyllogistic-in-unrhapsodically.ngrok-free.dev"

app = FastAPI()
print("Loading Whisper Model (Large-V3) on GPU...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Whisper Model Loaded!")

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/transcribe")
async def transcribe(file: UploadFile = File(...), language: str = Form("en")):
    try:
        path = f"/tmp/{file.filename}"
        with open(path, "wb") as f: f.write(await file.read())
        segments, _ = model.transcribe(path, language=None if language=="auto" else language)
        text = "".join([s.text for s in segments])
        os.remove(path)
        return {"text": text.strip(), "confidence": 0.9}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

@app.post("/summarize")
async def summarize(request: dict):
    try:
        prompt = request.get("prompt", "")
        model_name = request.get("model", "llama3.2")
        result = subprocess.run(
            ["ollama", "run", model_name, prompt],
            capture_output=True, text=True, timeout=120
        )
        return {"response": result.stdout.strip()}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

nest_asyncio.apply()
threading.Thread(target=run_uvicorn, daemon=True).start()
time.sleep(5)

from pyngrok import ngrok
try:
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8000, domain=NGROK_DOMAIN).public_url
    print("\n" + "="*60)
    print(f"🚀 PERMANENT SERVER READY!")
    print(f"🔗 URL: {public_url}")
    print(f"✅ Endpoints: /health  /transcribe  /summarize")
    print("="*60)
    while True: time.sleep(100)
except Exception as e:
    print(f"Tunnel Error: {e}")
